# Session 11: Advanced Retrieval with LangChain

## Learning Objectives:

- Understand and implement multiple retrieval strategies for RAG
- Compare naive, BM25, multi-query, parent-document, contextual compression, ensemble, and semantic chunking approaches
- Build RAG chains over a health and wellness knowledge base using LangChain and QDrant

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

---

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

> NOTE: Create a `.env` file in this directory with `OPENAI_API_KEY` and `COHERE_API_KEY` to avoid being prompted each time.

In [1]:
import os
import getpass
from dotenv import load_dotenv
import truststore

truststore.inject_into_ssl()

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Health and Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, stress management, habits, and common health concerns.

### Data Preparation

We'll load the wellness guide as a single document, then split it into smaller chunks using a `RecursiveCharacterTextSplitter` for our vector store. We also keep the raw (unsplit) document for use with the Parent Document Retriever and Semantic Chunker later.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/HealthWellnessGuide.txt")
raw_docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
wellness_docs = text_splitter.split_documents(raw_docs)

Let's verify our data was loaded and split correctly!

In [4]:
print(f"Raw documents: {len(raw_docs)}")
print(f"Split chunks: {len(wellness_docs)}")
print(f"\nExample chunk:\n{wellness_docs[0]}")

Raw documents: 1
Split chunks: 45

Example chunk:
page_content='The Personal Wellness Guide
A Comprehensive Resource for Health and Well-being

PART 1: EXERCISE AND MOVEMENT

Chapter 1: Understanding Exercise Basics

Exercise is one of the most important things you can do for your health. Regular physical activity can improve your brain health, help manage weight, reduce the risk of disease, strengthen bones and muscles, and improve your ability to do everyday activities.' metadata={'source': 'data/HealthWellnessGuide.txt'}


## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "wellness_guide".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = QdrantVectorStore.from_documents(
    wellness_docs,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide",
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [9]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- Cat-Cow Stretch: Begin on your hands and knees. Alternate between arching your back up (cat pose) and letting it sag down (cow pose). Perform 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg simultaneously, keeping your core engaged. Hold each for about 5 seconds, then switch sides. Aim for 10 repetitions per side.\n- Pelvic Tilts: Lie on your back with knees bent. Flatten your back against the floor by tightening your abdominal muscles and tilting your pelvis upward. Hold for 10 seconds and repeat 8-12 times.\n- Partial Crunches: Lie on your back with knees bent, arms crossed over your chest, and lift your shoulders off the floor slightly, engaging your stomach muscles. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat on the ground. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle and aimed

In [11]:
naive_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep significantly impacts overall health by supporting vital bodily functions. During sleep, the body repairs tissues, which is essential for physical health. Sleep also plays a key role in consolidating memories, contributing to mental well-being and cognitive function. Additionally, sleep regulates hormones related to growth and appetite, helping maintain a healthy weight and metabolic balance. Adequate sleep (7-9 hours per night) is associated with a stronger immune system, better emotional health, and increased longevity. Poor sleep or sleep disorders like insomnia can negatively affect these processes, leading to increased risks for various health issues. Therefore, maintaining good sleep hygiene and creating an optimal sleep environment are crucial for overall health and well-being.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gently massaging temples and neck\n- Using essential oils such as peppermint or lavender\n- Maintaining a regular sleep schedule\n- Practicing relaxation techniques like deep breathing and progressive muscle relaxation\n- Taking short walks, especially in nature\n- Listening to calming music\n- Engaging in mindfulness or meditation practices\n\nThese approaches can help manage stress and alleviate headache symptoms naturally.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(wellness_docs)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (cat) and letting it sag down (cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold each extension for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your back against the floor by tightening your abs and tilting your pelvis up slightly. Hold for 10 seconds and repeat 8-12 times.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and prevent future episodes.'

In [16]:
bm25_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a crucial role in overall health. It occurs in cycles that include REM and non-REM stages, each contributing to different aspects of physical and mental well-being. During deep sleep (Stage 3), the body repairs and regenerates tissues, supporting physical health. REM sleep is vital for brain activity related to memory, learning, and mental health. \n\nGetting 7-9 hours of quality sleep per night helps maintain a healthy immune system, supports cognitive function, and promotes emotional well-being. Conversely, insufficient or poor-quality sleep can impair these functions and may contribute to issues such as insomnia, fatigue, weakened immunity, and decreased mental clarity. Creating an optimal sleep environment—such as a comfortable mattress, darkness, and a cool temperature—can help improve sleep quality and, consequently, overall health.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include relaxation techniques such as deep breathing and progressive muscle relaxation, herbal teas like chamomile or valerian root, meditation, and mindfulness practices. Additionally, staying well-hydrated, managing sleep through relaxation methods, and addressing triggers like stress can help reduce headaches and stress levels.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### Answer:

A query like "Cat-Cow Stretch repetitions" or "Chapter 10" would benefit from BM25 over embedding-based retrieval. BM25 operates on exact keyword matching (term frequency / inverse document frequency), so when a query contains specific terms that appear in the target documents, BM25 will directly get the chunks containing those exact words. Embedding-based retrieval on the other hand, maps queries and documents into a shared semantic vector space — it is better at understanding meaning and synonyms, but it can miss the signal when the user is looking for a specific named entity or technical term. For instance, "Cat-Cow Stretch repetitions" has very precise vocabulary that appears word-for-word in the wellness guide; BM25 would rank the exact matching chunk highly, while the embedding model might also return semantically similar but less specific documents about other stretches. In general, BM25 shines when queries are keyword-heavy, contain proper nouns, or use domain-specific jargon that embedding models may not differentiate well from related concepts.

## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
import ssl
import truststore
import httpx
import cohere
from langchain_cohere import CohereRerank
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever

ctx = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
httpx_client = httpx.Client(verify=ctx, timeout=60.0)

co = cohere.ClientV2(api_key=os.environ["COHERE_API_KEY"], httpx_client=httpx_client)
compressor = CohereRerank(model="rerank-v3.5", client=co)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'To help with lower back pain, gentle stretching and strengthening exercises are recommended. Some specific exercises include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees. Alternate between arching your back up (like a cat) and letting it sag down (like a cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend your opposite arm and leg while keeping your core engaged. Hold the position for 5 seconds, then switch sides. Aim for 10 repetitions on each side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent. Flatten your lower back against the floor by tightening your abdominal muscles and tilting your pelvis slightly upward. Hold for 10 seconds and repeat 8-12 times.\n\nAlways consult with a healthcare professional before starting new exercises, especially if you have severe or persistent pain.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep has a significant impact on overall health. It is essential for physical health, mental well-being, and cognitive function. During sleep, the body repairs tissues, consolidates memories, and releases hormones that regulate growth and appetite. Adequate sleep, typically 7-9 hours per night for adults, occurs in cycles that include REM and non-REM stages, each playing a vital role in restoring and rejuvenating the body and brain. Creating a comfortable sleep environment and managing sleep issues like insomnia can further support these health benefits.'

In [23]:
contextual_compression_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include drinking water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark, quiet room, giving gentle massages to the temples and neck, using peppermint or lavender essential oils, maintaining a regular sleep schedule, practicing deep breathing, engaging in progressive muscle relaxation, using grounding techniques, taking short walks in nature, and listening to calming music.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
multi_query_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on your hands and knees, alternate between arching your back up (like a cat) and letting it sag down (like a cow). Perform 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend the opposite arm and leg while keeping your core engaged. Hold the position for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross your arms over your chest, tighten your stomach muscles, and lift your shoulders off the floor. Repeat 8-12 times.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot on the floor. Hold for 15-30 seconds and switch legs.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, tighten your abs, and tilt your pelvis upward to flatten your back against the floor. Hold for 10 seconds and repeat 8-12 times.\n\nThese exercises are gentle and aimed at strengthe

In [27]:
multi_query_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health. It supports physical recovery by helping the body repair tissues and regenerate cells. It also enhances mental well-being by consolidating memories and regulating hormones related to growth and appetite. Adequate sleep (7-9 hours for adults) is linked to improved immune function, better mood, cognitive function, and can reduce the risk of chronic conditions. Poor sleep or sleep disorders like insomnia can negatively impact physical health, increase stress levels, and impair emotional regulation, highlighting the importance of good sleep hygiene and routines for maintaining overall health.'

In [28]:
multi_query_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises (e.g., inhale for 4 counts, hold, exhale)\n- Progressive muscle relaxation, tensing and releasing muscle groups\n- Grounding techniques (naming things you see, hear, feel, smell, and taste)\n- Taking short walks, preferably in nature\n- Listening to calming music\n- Drinking water to stay hydrated\n- Applying cold or warm compresses to the head or neck\n- Resting in a dark, quiet room\n- Gentle massage of the temples and neck\n- Using essential oils like peppermint or lavender\n- Maintaining a regular sleep schedule\n\nThese methods can help alleviate stress and reduce headache symptoms naturally.'

### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### Answer:

Generating multiple reformulations of a user query improves recall by creating a wider semantic net over the document space. Each reformulation uses different vocabulary or phrasing which means different embedding vectors are produced and each vector may be closest to a different set of document chunks. For example, the original query "What are some natural remedies for stress and headaches?" might be reformulated as:

1. "How can I relieve stress naturally?"
2. "Home treatments for headache pain"

Each reformulation gets documents that may not have ranked highly for the original query alone. The Multi-Query Retriever then takes the union of all unique documents retrieved across all reformulations, which significantly increases the chance of capturing all relevant chunks (higher recall).

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. We split the full document into large "parent" chunks (e.g. 2000 characters).
2. Each parent chunk is further split into smaller "child" chunks (e.g. 400 characters).
3. The child chunks are stored in a VectorStore, while the parent chunks are stored in an in-memory docstore.
4. When we query our Retriever, we do a similarity search comparing our query vector to the child chunks.
5. Instead of returning the child chunks, we return their associated parent chunks.

The basic idea is:

- **Search** for small, focused chunks (better semantic matching)
- **Return** big chunks (richer surrounding context)

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by defining our parent and child splitters.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="wellness_parent_child",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="wellness_parent_child", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore=parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(raw_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
parent_document_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

'Exercises that can help with lower back pain include gentle stretching and strengthening movements such as:\n\n- Cat-Cow Stretch: Alternating between arching your back up and letting it sag down while on hands and knees (10-15 repetitions).\n- Bird Dog: Extending opposite arm and leg from a hands-and-knees position, holding for 5 seconds on each side (10 repetitions per side).\n- Partial Crunches: Lying on your back with knees bent, lifting shoulders off the floor while engaging your core (8-12 repetitions).\n- Knee-to-Chest Stretch: Pulling one knee toward your chest while lying on your back, holding for 15-30 seconds, then switching legs.\n- Pelvic Tilts: Flattening your lower back against the floor by engaging your abdominal muscles, holding for 10 seconds (8-12 repetitions).\n\nThese exercises can help improve flexibility, strengthen core muscles, and alleviate lower back discomfort. Always consult with a healthcare professional before starting new exercises, especially if you hav

In [35]:
parent_document_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical recovery, mental well-being, and cognitive functioning. During sleep, the body repairs tissues, consolidates memories, and releases hormones that help regulate growth and appetite. Adequate sleep—typically 7 to 9 hours for adults—ensures these processes occur effectively, which can improve mood, boost immunity, enhance concentration, and reduce the risk of chronic health conditions. Good sleep hygiene and creating a conducive sleep environment are important for maintaining quality sleep and overall health.'

In [36]:
parent_document_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include practicing deep breathing, engaging in relaxation exercises like progressive muscle relaxation, mindfulness meditation, and grounding techniques. Additionally, applying peppermint or lavender essential oils can help alleviate headache symptoms. Managing stress through regular exercise, maintaining a consistent sleep schedule, and spending time in nature or engaging in hobbies are also beneficial. Staying well-hydrated by drinking plenty of water can help reduce headaches triggered by dehydration.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
ensemble_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n\n- **Pelvic Tilts:** Lie on your back with knees bent, flatten your back against the floor by tightening your abs and tilting your pelvis slightly upward. Hold for 10 seconds, repeat 8-12 times.\n\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Hold briefly, then lower. Do 8-12 repetitions.\n\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n\nThese exercises are gentle and targeted to alleviate lower back discomfort. 

In [40]:
ensemble_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in overall health by supporting physical, mental, and cognitive functions. Adequate sleep—typically 7-9 hours per night—allows the body to repair tissues, regulate hormones related to growth and appetite, and consolidate memories. The quality of sleep, which depends on good sleep hygiene and a proper sleep environment, is crucial for maintaining immunity, managing stress, and preventing health issues such as insomnia. Poor sleep or sleep disturbances like insomnia can negatively impact mental health, increase fatigue, weaken the immune system, and contribute to chronic health problems. Therefore, prioritizing good sleep habits and environment is essential for overall wellness.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress include practicing deep breathing, progressive muscle relaxation, grounding techniques (such as naming objects you see, hear, feel, smell, and taste), taking short walks especially in nature, and listening to calming music. \n\nFor headaches, natural remedies include drinking plenty of water to stay hydrated, applying cold or warm compresses to the head or neck, resting in a dark and quiet room, gently massaging the temples and neck, using peppermint or lavender essential oils, and maintaining a regular sleep schedule. \n\nThese approaches can help manage stress and headaches effectively and naturally.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(raw_docs)

Let's create a new vector store.

In [44]:
semantic_vectorstore = QdrantVectorStore.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="wellness_guide_semantic_chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What exercises can help with lower back pain?"})["response"].content

"Exercises that can help with lower back pain include:\n\n- **Cat-Cow Stretch:** Start on hands and knees; alternate arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- **Partial Crunches:** Lie on your back with knees bent, cross arms over chest, tighten stomach muscles, and raise shoulders off the floor. Do 8-12 repetitions.\n- **Knee-to-Chest Stretch:** Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- **Pelvic Tilts:** Lie on your back with knees bent; flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.\n- **Bird Dog:** From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides.\n\nThese gentle stretching and strengthening exercises can help alleviate lower back discomfort and may prevent future episodes. If you have ongoing or severe pain

In [48]:
semantic_retrieval_chain.invoke({"question" : "How does sleep affect overall health?"})["response"].content

'Sleep plays a vital role in maintaining overall health. During sleep, the body undergoes tissue repair, hormone regulation, and memory consolidation. Adequate sleep, typically 7-9 hours per night for adults, is essential for physical health, mental well-being, and cognitive function. Good sleep quality supports immune function, helps manage weight through hormone regulation, reduces stress, and enhances mood. Conversely, poor sleep or sleep deprivation can lead to health issues such as increased stress, impaired immune response, weight gain, and mental health problems. Therefore, prioritizing healthy sleep habits and creating a restful sleep environment are crucial for overall health and wellness.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What are some natural remedies for stress and headaches?"})["response"].content

'Some natural remedies for stress and headaches include:\n\n- Deep breathing exercises: Inhale for 4 counts, hold for 4, exhale for 4, hold for 4.\n- Progressive muscle relaxation: Tense and release muscle groups systematically.\n- Grounding techniques: Name 5 things you see, 4 you hear, 3 you feel, 2 you smell, and 1 you taste.\n- Gentle walks, preferably outdoors or in nature.\n- Listening to calming music.\n- Hydrating adequately by drinking plenty of water.\n- Applying cold or warm compresses to the head or neck.\n- Resting in a dark, quiet room.\n- Using essential oils like peppermint or lavender.\n- Practicing mindfulness and meditation regularly.\n\nThese methods can help reduce stress and alleviate headache symptoms naturally.'

### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### Answer:

With short, highly repetitive sentences such as FAQs, semantic chunking would likely merge many unrelated Q&A pairs into a single oversized chunk. This happens because high inter sentence cosine similarity - FAQ sentences tend to share similar vocabulary and structure (e.g. "What is…?", "How do I…?"), so the embedding distance between consecutive sentences stays low. Another reason might be few breakpoints detected only - since the percentile threshold looks for unusually large distance jumps to split, and most FAQ sentences are similarly close in embedding space, the algorithm sees few breakpoints, resulting in very large chunks that hold together unrelated questions and answers.
I would adjust this by lowering the percentile treshold so that the smaller distance differences are enough to be split. Also, I would try switching to 'gradient' or 'standard_deviation' thresholding to see if the results improve.

---

# 🤝 Breakout Room Part #2

### 🏗️ Activity #1:

Your task is to evaluate the various Retriever methods against each other.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparison between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [51]:
# use sgd to create the dataset
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_31260\981573009.py:12: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_31260\981573009.py:13: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())


Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/5 [00:00<?, ?it/s]

...

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [52]:
df = dataset.to_pandas()
df = df.drop(columns=["persona_name", "query_style", "query_length"], errors="ignore")
df

,user_input,reference_contexts,reference,synthesizer_name
0,Whaat is the chaptar 1 about in exercize and m...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,Chapter 1: Understanding Exercise Basics expla...,single_hop_specific_query_synthesizer
1,what is part 1 in exercise and movement and wh...,[PART 1: EXERCISE AND MOVEMENT\n\nChapter 1: U...,PART 1: EXERCISE AND MOVEMENT explains that ex...,single_hop_specific_query_synthesizer
2,How does the use of white noise contribute to ...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,The context mentions that to create an optimal...,single_hop_specific_query_synthesizer
3,How do vegetables contribute to a balanced die...,[PART 2: NUTRITION AND DIET\n\nChapter 4: Fund...,Vegetables are a key component of a balanced d...,single_hop_specific_query_synthesizer
4,What is Chapter 20 about?,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 20 discusses social connections and he...,single_hop_specific_query_synthesizer
5,What is Chapter 13 about in the context of bui...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,Chapter 13 discusses the science of habit form...,single_hop_specific_query_synthesizer
6,How does Chapter 19 on work-life balance relat...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Chapter 19 emphasizes maintaining a healthy wo...,multi_hop_specific_query_synthesizer
7,Chapter 10 stress bad or good?,[<1-hop>\n\nPART 4: STRESS MANAGEMENT AND MENT...,Chapter 10 explains stress is body's response ...,multi_hop_specific_query_synthesizer
8,Wht is part 3 about?,[<1-hop>\n\nPART 2: NUTRITION AND DIET\n\nChap...,"Part 3 covers sleep and recovery, including th...",multi_hop_specific_query_synthesizer
9,Chapter 15 talk about building habits and also...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Chapter 15 discusses building healthy habits b...,multi_hop_specific_query_synthesizer


In [54]:
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
import copy, time

custom_run_config = RunConfig(timeout=360)
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

retrievers = {
    "naive": naive_retriever,
    "bm25": bm25_retriever,
    "multi_query": multi_query_retriever,
    "parent_doc": parent_document_retriever,
    "compression": compression_retriever,
    "ensemble": ensemble_retriever,
}

metrics = [
    LLMContextRecall(),
    ContextPrecision(),
    ContextEntityRecall()
]

results = {}

for name, retriever in retrievers.items():
    dataset_copy = df.copy()

    retrieved_contexts = []
    for _, row in dataset_copy.iterrows():
        q = row["user_input"]  # <- DataFrame column, not row.eval_sample.user_input

        # use whichever your retriever supports
        docs = retriever.invoke(q) if hasattr(retriever, "invoke") else retriever.get_relevant_documents(q)

        retrieved_contexts.append([d.page_content for d in docs])
        time.sleep(60)

    dataset_copy["retrieved_contexts"] = retrieved_contexts

    # Some Ragas versions expect response to exist even for retriever metrics
    if "response" not in dataset_copy.columns:
        dataset_copy["response"] = ""

    eval_ds = EvaluationDataset.from_pandas(dataset_copy)
    results[name] = evaluate(
        dataset=eval_ds,
        metrics=metrics,
        llm=evaluator_llm,
        run_config=custom_run_config
    )
    time.sleep(5)  # to avoid hitting rate limits

results

C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_31260\1713543821.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_31260\1713543821.py:2: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import LLMContextRecall, ContextEntityRecall, ContextPrecision, NoiseSensitivity
C:\Users\llukacevic2\AppData\Local\Temp\ipykernel_31260\1713543821.py:2: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ra

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/33 [00:00<?, ?it/s]

Exception raised in Job[10]: TimeoutError()
Exception raised in Job[7]: TimeoutError()
Exception raised in Job[1]: TimeoutError()
Exception raised in Job[4]: TimeoutError()
Exception raised in Job[13]: TimeoutError()
Exception raised in Job[16]: TimeoutError()
Exception raised in Job[22]: TimeoutError()


{'naive': {'context_recall': 0.7727, 'context_precision': 0.6672, 'context_entity_recall': 0.2535},
 'bm25': {'context_recall': 0.3939, 'context_precision': 0.5758, 'context_entity_recall': 0.0915},
 'multi_query': {'context_recall': 0.7727, 'context_precision': 0.6621, 'context_entity_recall': 0.2389},
 'parent_doc': {'context_recall': 0.8030, 'context_precision': 0.7803, 'context_entity_recall': 0.1985},
 'compression': {'context_recall': 0.5606, 'context_precision': 0.7879, 'context_entity_recall': 0.2237},
 'ensemble': {'context_recall': 0.9394, 'context_precision': 0.5652, 'context_entity_recall': 0.1728}}

In [66]:
# enable langsmith tracing

import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
os.environ["LANGCHAIN_PROJECT"] = "ai_bootcamp_session11"
os.environ["LANGCHAIN_ENDPOINT"] = "https://eu.api.smith.langchain.com"
# (Optional, also set the LangSmith-style name for consistency)
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

In [69]:
from langchain_community.callbacks import get_openai_callback
import time

chains = {
    "naive":       naive_retrieval_chain,
    "bm25":        bm25_retrieval_chain,
    "multi_query": multi_query_retrieval_chain,
    "parent_doc":  parent_document_retrieval_chain,
    "compression": contextual_compression_retrieval_chain,
    "ensemble":    ensemble_retrieval_chain,
}

questions = df["user_input"].tolist()
references = df["reference"].tolist()
ref_contexts = df["reference_contexts"].tolist()

all_results = []

for i, q in enumerate(questions):
    print(f"Question {i+1}/{len(questions)}: {q[:60]}...")
    row_data = {
        "question": q,
        "reference": references[i],
        "reference_contexts": ref_contexts[i],
    }

    for chain_name, chain in chains.items():
        try:
            # get_openai_callback tracks tokens + cost for all OpenAI calls inside
            with get_openai_callback() as cb:
                t0 = time.perf_counter()
                result = chain.invoke({"question": q})
                latency = time.perf_counter() - t0

            row_data[f"{chain_name}_latency_s"] = round(latency, 3)
            row_data[f"{chain_name}_total_cost_usd"] = round(cb.total_cost, 6)
            row_data[f"{chain_name}_prompt_tokens"] = cb.prompt_tokens
            row_data[f"{chain_name}_completion_tokens"] = cb.completion_tokens
            row_data[f"{chain_name}_total_tokens"] = cb.total_tokens
            row_data[f"{chain_name}_response"] = result["response"].content
            print(f"  ✅ {chain_name}: {latency:.2f}s, ${cb.total_cost:.6f}, {cb.total_tokens} tokens")
        except Exception as e:
            row_data[f"{chain_name}_latency_s"] = None
            row_data[f"{chain_name}_total_cost_usd"] = None
            row_data[f"{chain_name}_prompt_tokens"] = None
            row_data[f"{chain_name}_completion_tokens"] = None
            row_data[f"{chain_name}_total_tokens"] = None
            row_data[f"{chain_name}_response"] = f"ERROR: {e}"
            print(f"  ⚠️ {chain_name}: {e}")

        # 7s sleep to stay under Cohere's 10 calls/min trial limit
        time.sleep(7)

    all_results.append(row_data)

print(f"\n✅ Done! Collected data for {len(all_results)} questions × {len(chains)} chains")

Question 1/11: Whaat is the chaptar 1 about in exercize and movment?...
  ✅ naive: 2.36s, $0.000161, 1422 tokens
  ✅ bm25: 1.50s, $0.000084, 635 tokens
  ✅ multi_query: 4.16s, $0.000232, 1995 tokens
  ✅ parent_doc: 1.70s, $0.000118, 1003 tokens
  ✅ compression: 2.21s, $0.000065, 501 tokens
  ✅ ensemble: 6.01s, $0.000371, 3263 tokens
Question 2/11: what is part 1 in exercise and movement and why is it import...
  ✅ naive: 1.97s, $0.000187, 1501 tokens
  ✅ bm25: 1.41s, $0.000079, 550 tokens
  ✅ multi_query: 3.67s, $0.000257, 2082 tokens
  ✅ parent_doc: 2.32s, $0.000147, 1075 tokens
  ✅ compression: 2.70s, $0.000082, 578 tokens
  ✅ ensemble: 6.48s, $0.000395, 3433 tokens
Question 3/11: How does the use of white noise contribute to improving slee...
  ✅ naive: 1.86s, $0.000206, 1705 tokens
  ✅ bm25: 1.81s, $0.000099, 644 tokens
  ✅ multi_query: 4.57s, $0.000287, 2309 tokens
  ✅ parent_doc: 2.17s, $0.000099, 652 tokens
  ✅ compression: 2.37s, $0.000089, 645 tokens
  ✅ ensemble: 6.59s, $0.00

In [73]:
import pandas as pd

summary = []
for name in list(chains.keys()):
    costs = [r[f"{name}_total_cost_usd"] for r in all_results if r.get(f"{name}_total_cost_usd") is not None]
    latencies = [r[f"{name}_latency_s"] for r in all_results if r.get(f"{name}_latency_s") is not None]
    tokens = [r[f"{name}_total_tokens"] for r in all_results if r.get(f"{name}_total_tokens") is not None]
    summary.append({
        "retriever": name,
        "avg_latency_s": round(sum(latencies) / len(latencies), 3) if latencies else None,
        "avg_cost_usd": round(sum(costs) / len(costs), 6) if costs else None,
        "total_cost_usd": round(sum(costs), 6) if costs else None,
        "avg_tokens": round(sum(tokens) / len(tokens)) if tokens else None,
        "success_rate": f"{len(costs)}/{len(all_results)}",
    })

pd.DataFrame(summary)

,retriever,avg_latency_s,avg_cost_usd,total_cost_usd,avg_tokens,success_rate
0,naive,2.395,0.000184,0.002026,1536,11/11
1,bm25,1.757,0.000089,0.000982,597,11/11
2,multi_query,4.132,0.000268,0.002947,2172,11/11
3,parent_doc,2.502,0.000149,0.001639,1116,11/11
4,compression,2.604,0.000086,0.000942,597,11/11
5,ensemble,6.140,0.000385,0.004236,3289,11/11


{'naive': {'context_recall': 0.7727, 'context_precision': 0.6672, 'context_entity_recall': 0.2535},
 'bm25': {'context_recall': 0.3939, 'context_precision': 0.5758, 'context_entity_recall': 0.0915},
 'multi_query': {'context_recall': 0.7727, 'context_precision': 0.6621, 'context_entity_recall': 0.2389},
 'parent_doc': {'context_recall': 0.8030, 'context_precision': 0.7803, 'context_entity_recall': 0.1985},
 'compression': {'context_recall': 0.5606, 'context_precision': 0.7879, 'context_entity_recall': 0.2237},
 'ensemble': {'context_recall': 0.9394, 'context_precision': 0.5652, 'context_entity_recall': 0.1728}}

### Analysis

**Performance**: The **Ensemble Retriever** achieves the highest context recall (0.9394), meaning it finds the most relevant information across queries — this makes sense as it combines all five retriever strategies via Reciprocal Rank Fusion. However, its context precision is the lowest (0.5652), indicating it also retrieves more noise. The **Parent-Document Retriever** strikes the best balance between recall (0.8030) and precision (0.7803), delivering relevant and focused context by searching small child chunks but returning their richer parent chunks. **Compression (Rerank)** achieves the highest precision (0.7879) but sacrifices recall (0.5606), since the Cohere reranker aggressively filters down to only the most relevant documents.

**Cost**: **BM25** and **Compression** are the cheapest at ~$0.000089 and ~$0.000086 per query respectively. BM25 is cheap because it requires no embedding or LLM calls for retrieval. Compression is cheap in OpenAI terms because the reranker reduces the context size passed to the LLM — though this does not account for the Cohere reranker API cost. **Ensemble** is the most expensive ($0.000385/query) because it runs all five retrievers plus the LLM on a large combined context.

**Latency**: **BM25** is the fastest (1.76s avg) due to its purely local, keyword-based matching. **Naive** and **Parent-Doc** are in the middle (~2.4–2.5s). **Multi-Query** is slower (4.13s) because it makes an extra LLM call to generate reformulations. **Ensemble** is the slowest (6.14s) as it must run all retrievers sequentially.

For this particular Health & Wellness dataset, the **Parent-Document Retriever** offers the best overall trade-off:

- **Strong performance**: Highest combined recall + precision balance (0.80 recall, 0.78 precision)
- **Moderate cost**: $0.000149/query — 39% cheaper than multi-query, 61% cheaper than ensemble
- **Reasonable latency**: 2.50s — only slightly slower than naive (2.40s)

The "small-to-big" strategy works particularly well on this corpus because the wellness guide has clear topical structure (chapters and sections). Child chunks capture specific details (e.g. "Cat-Cow Stretch: 10-15 reps") while parent chunks provide the surrounding context about the broader topic.